In [ ]:
import pandas as pd
import os
import numpy as np
from tqdm import tqdm
# Load the data
data_path = os.path.join(os.getcwd(), "data", "proid_unique_20231017.dta")
proid_unique = pd.read_stata(data_path)

# Check the data
print(proid_unique.head())
print(proid_unique.dtypes)

In [ ]:
print(proid_unique.proid.nunique())

In [3]:
# load the id2firms_all.txt file to dataframe
id2firm_path = os.path.join(os.getcwd(), "data", "input", "id2firms_all.txt")
id2firms = pd.read_csv(id2firm_path, sep='\t', header=0)

In [ ]:
print(id2firms.dtypes)
print(id2firms.proid.nunique())
id2firms.head(100)

In [5]:
# convert the proid column to int
id2firms['proid'].fillna(-999, inplace=True)
id2firms['proid'] = id2firms['proid'].astype(int)
id2firms = pd.merge(id2firms, proid_unique, on='proid', how='left')

In [ ]:
print(pd.isnull(id2firms[['emaskcd', 'amaskcd']]).sum())
id2firms.head()

In [7]:
# load the Allstar analysts data
all_star_path = os.path.join(os.getcwd(), "data", "Allstar.dta")
all_star = pd.read_stata(all_star_path)

In [ ]:
# merge the all_star data with the id2firms data
all_star.columns = ['year', 'emaskcd', 'amaskcd']
all_star['all_star'] = 1
id2firms = pd.merge(id2firms, all_star, on=['year', 'emaskcd', 'amaskcd'], how='left', indicator=True)
id2firms.head()

In [9]:
analyst_exp_path = os.path.join(os.getcwd(), "data", "analyst_experience_fyear_20220611.dta")
analyst_exp = pd.read_stata(analyst_exp_path)

In [ ]:
analyst_exp.head()

In [ ]:
analyst_exp_sub = analyst_exp.drop(columns=['gvkey'])
analyst_exp_sub.drop_duplicates(inplace=True)
print(pd.isnull(analyst_exp_sub).sum())
print(analyst_exp_sub.nunique())

In [ ]:
# the check the number of unique variables by column year, amaskcd, emaskcd and by column amaskcd
print(analyst_exp_sub.drop_duplicates(subset=['fyear', 'AMASKCD', 'EMASKCD']).shape)
print(analyst_exp_sub.drop_duplicates(subset=['AMASKCD', 'GenExp', 'FirmExp']).shape)
print(analyst_exp_sub.drop_duplicates(subset=['fyear','AMASKCD', 'EMASKCD', 'GenExp', 'FirmExp']).shape)
print(analyst_exp_sub.drop_duplicates(subset=['fyear','AMASKCD', 'EMASKCD', 'GenExp']).shape)
print(analyst_exp_sub.drop_duplicates(subset=['fyear','AMASKCD', 'EMASKCD']).shape)

print(id2firms.drop_duplicates(subset=['year', 'amaskcd', 'emaskcd']).shape)
# print(id2firms.drop_duplicates(subset=['year', 'amaskcd', 'emaskcd']).nunique())
# print(analyst_exp_sub.drop_duplicates(subset=['fyear', 'AMASKCD', 'EMASKCD']).nunique())
# print(analyst_exp_sub.drop_duplicates(subset=['AMASKCD', 'GenExp', 'FirmExp', 'Broker_size']).nunique())
print(id2firms.drop_duplicates(subset=['amaskcd']).shape)

In [ ]:
# Alternative approach with more careful data preparation
# 1. Reset index before merge
id2firms = id2firms.reset_index(drop=True)
analyst_exp_sub = analyst_exp_sub.reset_index(drop=True)
analyst_exp_sub.rename(columns={'fyear': 'year', 'AMASKCD': 'amaskcd', 'EMASKCD': 'emaskcd'}, inplace=True)

# 2. Ensure consistent data types
merge_columns = ['year', 'amaskcd', 'emaskcd']
for col in merge_columns:
    id2firms[col] = pd.to_numeric(id2firms[col], errors='coerce')
    analyst_exp_sub[col] = pd.to_numeric(analyst_exp_sub[col], errors='coerce')

# 3. Remove any duplicates in analyst_exp_sub
analyst_exp_sub = analyst_exp_sub.drop_duplicates(subset=merge_columns)
id2firms.drop(columns=['_merge'], inplace=True)

# 4. Try merge again
try:
    id2firms_alyst = pd.merge(
        id2firms,
        analyst_exp_sub,
        on=merge_columns,
        how='left',
        indicator=True
    )
    
    print("Merge successful!")
    print("Rows in final dataset:", len(id2firms_alyst))
    print("Merge results:", id2firms_alyst['_merge'].value_counts())
    
except Exception as e:
    print(f"Merge failed with error: {e}")

In [ ]:
id2firms_alyst.head()
1/0

In [ ]:
id2firms_alyst._merge.value_counts()

In [ ]:
#please check the number of missing values in the columns of proid in id2firms_alyst
print("Number of missing values in GenExp:", id2firms_alyst['GenExp'].isna().sum())
# please check the number of proid as the value of -999
# print(id2firms_alyst['GenExp'].value_counts().to_frame().T)

In [ ]:
# view how many analysts I successfully merge
# view how many analysts I merged but with no experience
print(id2firms_alyst[id2firms_alyst._merge == 'both'].nunique().to_frame().T)

In [ ]:
# Sort the dataframe first for better groupby performance
id2firms_alyst.sort_values(by=['gvkey', 'year', 'quarter'], inplace=True)

# Use groupby with agg to check if all rows in group are 'left_only'
grouped = (id2firms_alyst.groupby(['gvkey', 'year', 'quarter'])['_merge']
           .agg(lambda x: (x != 'both').all())
           .reset_index())

# Count groups where all rows are 'left_only'
no_match_count = grouped['_merge'].sum()

print(f"Number of groups with no matches: {no_match_count}")
print(f"Total number of groups: {len(grouped)}")
print(f"Percentage of groups with no matches: {(no_match_count/len(grouped))*100:.2f}%")

In [ ]:
# 1. First get the groups that have at least one 'both'
groups_with_both = (id2firms_alyst.groupby(['gvkey', 'year', 'quarter'])['_merge']
                   .agg(lambda x: (x == 'both').any())
                   .reset_index())

# 2. Filter to keep only those groups
groups_to_keep = groups_with_both[groups_with_both['_merge']]

# 3. Merge back to get the filtered dataframe
id2firms_alyst_filtered = pd.merge(
    id2firms_alyst,
    groups_to_keep[['gvkey', 'year', 'quarter']],
    on=['gvkey', 'year', 'quarter'],
    how='inner'
)

# Print results
print(f"Original shape: {id2firms_alyst.shape}")
print(f"Filtered shape: {id2firms_alyst_filtered.shape}")
print(f"Groups removed: {len(groups_with_both) - len(groups_to_keep)}")

# Verify the results
print("\nMerge indicator counts in filtered data:")
print(id2firms_alyst_filtered['_merge'].value_counts())

In [ ]:
id2firms_alyst_filtered.head()

In [ ]:
id2firms_alyst.drop(columns=['_merge'], inplace=True)
id2firms_alyst = pd.merge(id2firms_alyst, id2firms_alyst_filtered[['gvkey', 'year', 'quarter']], on=['gvkey', 'year', 'quarter'], how='left', indicator=True)
id2firms_alyst.rename(columns={'_merge': 'analyst_match'}, inplace=True)

In [ ]:
# save the id2firms_alyst_filtered to a csv file
id2firms_alyst_filtered.to_csv(os.path.join(os.getcwd(), "data", "id2firms_alyst_filtered.csv"), index=False)
# save the id2firms_alyst to a csv file
id2firms_alyst.to_csv(os.path.join(os.getcwd(), "data", "id2firms_alyst.csv"), index=False)
# save the id2firms to a csv file
id2firms.to_csv(os.path.join(os.getcwd(), "data", "id2firms.csv"), index=False)

In [1]:
import pandas as pd
import os
import numpy as np
from tqdm import tqdm
# read the id2firms_alyst data
id2firms_alyst = pd.read_csv(os.path.join(os.getcwd(), "data", "id2firms_alyst.csv"))
# read the id2firms_
id2firms_alyst.head()

,transcriptid,companyid,gvkey,year,quarter,date,transcriptcomponenttypename,sentenceid,componentorder,proid,transcriptpersonname,word_count,country1,emaskcd,amaskcd,all_star,GenExp,FirmExp,Broker_size,_merge
0,1285935.0,875770,101434.0,2017.0,3,2017-08-23,Question,51961702.0,36.0,29072560,Matthias Pfeifenberger,37.0,Austria,1382.0,115607.0,NaN,NaN,NaN,NaN,left_only
1,1285980.0,875770,101434.0,2017.0,3,2017-08-23,Question and Answer Operator Message,51963412.0,36.0,-999,Operator,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,1285935.0,875770,101434.0,2017.0,3,2017-08-23,Answer,51961703.0,37.0,302796741,Stefan Doboczky,136.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,1285980.0,875770,101434.0,2017.0,3,2017-08-23,Question,51963413.0,37.0,29072560,Matthias Pfeifenberger,39.0,Austria,1382.0,115607.0,NaN,NaN,NaN,NaN,left_only
4,1285980.0,875770,101434.0,2017.0,3,2017-08-23,Answer,51963414.0,38.0,302796741,Stefan Doboczky,149.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [2]:
def expanding_quintile(df, var_names):
    """
    Label series in quintiles using expanding window
    
    Parameters:
    df: pd.DataFrame - dataframe containing the data
    var_names: list - [date_column, value_column]
    
    Returns:
    pd.DataFrame - original dataframe with new quintile column
    """
    # Sort by date
    df.sort_values(by=var_names[0], inplace=True)
    
    # Initialize results series
    quintile_labels = pd.Series(index=df.index, dtype=float)
    
    # Label each point using only historical data
    for i in tqdm(range(len(df)), desc="Labeling in Quintiles", bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}', colour="green"):
        historical_data = df[var_names[1]].iloc[:i+1]
        if len(historical_data) >= 5:  # Need at least 5 points to make quintiles
            try:
                quintile_labels.iloc[i] = pd.qcut(historical_data, 5, labels=False).iloc[-1]
            except ValueError as e:
                print(f"Error at index {i}: {e}")
                print(f"Data sample: {historical_data.value_counts()}")
                quintile_labels.iloc[i] = np.nan
        else:
            quintile_labels.iloc[i] = np.nan
    
    # Add the quintile labels to the dataframe
    df[f'{var_names[1]}_q5'] = quintile_labels
    
    return df

In [3]:
def top_bottom_quintile(df, var_name):
    """
    Compute the top and bottom quintile for analyst experience
    
    Parameters:
    df: DataFrame with columns ['proid', 'year', var_name]
    var_name: str, name of the variable to compute quintiles for (e.g., 'GenExp')
    
    Returns:
    Series with quintile labels
    """
    # Get the last observation for each proid-year combination
    temp = (df.groupby(['proid', 'year'])[var_name]
             .last()
             .reset_index()
             .sort_values(var_name, ascending=False))
    
    # Create quintiles and return only the quintile column
    quintiles = pd.qcut(temp[var_name], 5, labels=False)
    
    # Create a mapping dictionary from (proid, year) to quintile
    quintile_map = dict(zip(zip(temp['proid'], temp['year']), quintiles))
    
    # Map the quintiles back to original DataFrame index
    return df.apply(lambda x: quintile_map.get((x['proid'], x['year'])), axis=1)

# Apply the function
id2firms_alyst['GenExp_q5'] = top_bottom_quintile(id2firms_alyst, 'GenExp')

# Verify results
print("Sample of results:")
print(id2firms_alyst[['proid', 'year', 'GenExp', 'GenExp_q5']].head())
print("\nQuintile distribution:")
print(id2firms_alyst['GenExp_q5'].value_counts(normalize=True).sort_index())

Sample of results:
       proid    year  GenExp  GenExp_q5
0   29072560  2017.0     NaN        NaN
1       -999  2017.0     NaN        NaN
2  302796741  2017.0     NaN        NaN
3   29072560  2017.0     NaN        NaN
4  302796741  2017.0     NaN        NaN

Quintile distribution:
0.0    0.135526
1.0    0.206907
2.0    0.195474
3.0    0.222155
4.0    0.239938
Name: GenExp_q5, dtype: float64


In [4]:
# Diagnostic checks on your data
print("Original data shape:", id2firms_alyst.shape)
print("\nUnique values:")
print("proid:", id2firms_alyst['proid'].nunique())
print("year:", id2firms_alyst['year'].nunique())
print("\nMissing values in key columns:")
print(id2firms_alyst[['proid', 'year', 'GenExp']].isna().sum())

# Check the distribution of GenExp
print("\nGenExp distribution:")
print(id2firms_alyst['GenExp'].describe())

# Check groupby results
grouped = id2firms_alyst.groupby(['proid', 'year'])['GenExp'].last().reset_index()
print("\nAfter groupby shape:", grouped.shape)
print("Missing values after groupby:", grouped['GenExp'].isna().mean() * 100, "%")

Original data shape: (6727338, 21)

Unique values:
proid: 64786
year: 17

Missing values in key columns:
proid           0
year            0
GenExp    5953067
dtype: int64

GenExp distribution:
count    774271.000000
mean         14.077348
std           9.996644
min           1.000000
25%           6.000000
50%          11.000000
75%          22.000000
max          40.000000
Name: GenExp, dtype: float64

After groupby shape: (195292, 3)
Missing values after groupby: 86.18376584806342 %


In [5]:
def top_bottom_quintile1(df, var_name):
    """
    Compute the top and bottom quintile for analyst experience
    
    Parameters:
    df: DataFrame with columns ['proid', 'year', var_name]
    var_name: str, name of the variable to compute quintiles for (e.g., 'GenExp')
    
    Returns:
    Series with quintile labels
    """
    # Get the last observation for each proid-year combination
    temp = (df.groupby(['proid', 'year'])[var_name]
             .last()
             .reset_index()
             .sort_values(var_name, ascending=False))
    
    # Create quintiles and handle duplicate values with 'drop' option
    try:
        quintiles = pd.qcut(temp[var_name], 5, labels=False, duplicates='drop')
    except ValueError:
        # If we still get an error, try an alternative approach with rank
        quintiles = pd.Series(temp[var_name].rank(method='first'))
        quintiles = pd.qcut(quintiles, 5, labels=False)
    
    # Create a mapping dictionary from (proid, year) to quintile
    quintile_map = dict(zip(zip(temp['proid'], temp['year']), quintiles))
    
    # Map the quintiles back to original DataFrame index
    return df.apply(lambda x: quintile_map.get((x['proid'], x['year'])), axis=1)

In [6]:
import gc
gc.collect()
id2firms_alyst['FirmExp_q5'] = top_bottom_quintile1(id2firms_alyst, 'FirmExp')
id2firms_alyst.head()

,transcriptid,companyid,gvkey,year,quarter,date,transcriptcomponenttypename,sentenceid,componentorder,proid,...,country1,emaskcd,amaskcd,all_star,GenExp,FirmExp,Broker_size,_merge,GenExp_q5,FirmExp_q5
0,1285935.0,875770,101434.0,2017.0,3,2017-08-23,Question,51961702.0,36.0,29072560,...,Austria,1382.0,115607.0,NaN,NaN,NaN,NaN,left_only,NaN,NaN
1,1285980.0,875770,101434.0,2017.0,3,2017-08-23,Question and Answer Operator Message,51963412.0,36.0,-999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,NaN,NaN
2,1285935.0,875770,101434.0,2017.0,3,2017-08-23,Answer,51961703.0,37.0,302796741,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,NaN,NaN
3,1285980.0,875770,101434.0,2017.0,3,2017-08-23,Question,51963413.0,37.0,29072560,...,Austria,1382.0,115607.0,NaN,NaN,NaN,NaN,left_only,NaN,NaN
4,1285980.0,875770,101434.0,2017.0,3,2017-08-23,Answer,51963414.0,38.0,302796741,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,NaN,NaN


In [12]:
# id2firms_alyst[['transcriptcomponenttypename']].value_counts()
# drop rows with transcriptcomponenttypename not in ['Presenter Speech', 'Question', 'Answer']
id2firms_alyst = id2firms_alyst[id2firms_alyst['transcriptcomponenttypename'].isin(['Presenter Speech', 'Question', 'Answer'])]
# drop transcriptcomponenttypename column
id2firms_alyst[['transcriptcomponenttypename']].value_counts()


transcriptcomponenttypename
Answer                         3023506
Question                       2284860
Presenter Speech                416250
dtype: int64

In [7]:
# Diagnostic checks on your data
print("Original data shape:", id2firms_alyst.shape)
print("\nUnique values:")
print("proid:", id2firms_alyst['proid'].nunique())
print("year:", id2firms_alyst['year'].nunique())
print("\nMissing values in key columns:")
print(id2firms_alyst[['proid', 'year', 'FirmExp']].isna().sum())

# Check the distribution of GenExp
print("\nGenExp distribution:")
print(id2firms_alyst['FirmExp'].describe())

# Check groupby results
grouped = id2firms_alyst.groupby(['proid', 'year'])['FirmExp'].last().reset_index()
print("\nAfter groupby shape:", grouped.shape)
print("Missing values after groupby:", grouped['FirmExp'].isna().mean() * 100, "%")

Original data shape: (6727338, 22)

Unique values:
proid: 64786
year: 17

Missing values in key columns:
proid            0
year             0
FirmExp    5953067
dtype: int64

GenExp distribution:
count    774271.000000
mean          5.592306
std           5.208903
min           1.000000
25%           2.000000
50%           4.000000
75%           8.000000
max          38.000000
Name: FirmExp, dtype: float64

After groupby shape: (195292, 3)
Missing values after groupby: 86.18376584806342 %


In [13]:
# save the id2firms_alyst to a csv file
id2firms_alyst.to_csv(os.path.join(os.getcwd(), "data", "id2firms_alyst.csv"), index=False)

In [18]:
# write a code to compute the Topic Attention Divergence for each firm in id2firms (TAD)
# design the optimal algorithm to compute the 
# write a function to load the narratives data by sentence_id, 
# filepath = /outputs/scores/TF/combined_scores_TF.csv narratives_path = os.path.join(os.getcwd(), "w2v_culture", "outputs", "scores", "TF", "combined_scores_TF.csv")
# load the narratives data in the function in chunks of 100000 rows
# write a function to compute the TAD for each firm in id2firms
# the function should take the narratives data and the firm_id as input
# the function should return the TAD for the firm

# standardize the narratives data by divide the score based on the 
from tqdm import tqdm

class TAD:
    def __init__(self, model = 'TFIDF'):
        self.narratives_path = os.path.join(os.getcwd(), "w2v_culture", "outputs", "scores", f"{model}", f"combined_scores_{model}.csv")
        self.word_contribution_path = os.path.join(os.getcwd(), "w2v_culture", "outputs", "scores", "word_contribution")
        self.narratives = self.load_narratives()
        self.word_contributions, self.word_contribution_len = self.read_word_contribution()
        self.model_type = model
        self.topics_ = self.word_contribution.keys().tolist()
        self.analyst_feature = "GenExp"

    def load_narratives(self):
        # load the narratives data in the function in chunks of 100000 rows
        narratives = pd.read_csv(self.narratives_path, chunksize=100000)
        return narratives

    def merge_narratives_with_id2firms(self, id2firms):
        # merge the narratives data with the id2firms data
        id2firms = pd.merge(id2firms, self.narratives, on='sentence_id', how='left')
        # standardize the narratives data by divide the score based on the 
        return id2firms
    
    def read_word_contribution(self):
        # Read all CSV files from the word contributions directory to a dictionary
        # input: the word contribution path, with a topic folder for each topic csv file
        # output: a dictionary with the topic as the key and the dataframe as the value, 
        # output2: a list with the topic as key and list of length of each topic
        word_contributions = {}
        word_contribution_len = []
        for topic in os.listdir(self.word_contribution_path):
            topic_path = os.path.join(self.word_contribution_path, topic)
            if os.path.isdir(topic_path):  # Check if it's a directory
                for file in os.listdir(topic_path):
                    if file.endswith('.csv') and f'{self.model_type}' in file:
                        file_path = os.path.join(topic_path, file)
                        df = pd.read_csv(file_path)
                        df.columns = ['words', 'words_weight']
                        word_contributions[topic] = df
                        word_contribution_len.append(len(df))
            else:
                print(f"Skipping non-directory item: {topic_path}")
        return word_contributions, word_contribution_len
    

    def compute_cosine_similarity(self, v1, v2):
        # compute the cosine similarity between two vectors
        # input: two vectors
        # output: the cosine similarity
        return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    
    def get_v_a_con(self, row_doc, section_id='Ptranscriptcomponenttypename'):
        # get the vector for each firm_id, year, quarter (conference call level)
        # input: the row_doc each row is a conference call, firm_id, year, quarter 
        # add weight to vector based on analyst features
        row_doc = self.add_filter_analyst_feature(row_doc, self.analyst_feature)
        grouped_vectors = (row_doc.groupby([section_id])[self.topics_].mean()
                    .reindex(['Presenter Speech', 'Question', 'Answer']))
        
        # convert the temp_doc to a list of vectors, vector of presenterspeech, question, answer
        ps_vector = grouped_vectors.loc['Presenter Speech'].values
        q_vector = grouped_vectors.loc['Question'].values
        a_vector = grouped_vectors.loc['Answer'].values
        
        # get the cosine_similarity for each document
        tad_ps_q = 1 - self.compute_cosine_similarity(ps_vector, q_vector)
        tad_ps_a = 1 - self.compute_cosine_similarity(ps_vector, a_vector)
        tad_q_a = 1 - self.compute_cosine_similarity(q_vector, a_vector)
        
        return tad_ps_q, tad_ps_a, tad_q_a

    def cpt_firm_TAD(self, df, firm_id = 'gvkey'):
        # compute the cosine similarity between the word contribution and the narratives
        # input: the id2firms data
        # keys: the keys to group the data by firm_id, year and quarter, and section_id 
        # output: the cosine similarity for each firm
        # Get word contributions and their lengths
        keys = [f'{firm_id}', 'year', 'quarter']
        grouped = df.groupby(keys)
        result = []
        # process each document
        for names, group in tqdm(grouped, desc="Computing Cosine Similarity", bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}', colour="green"):
            # get the vector for each firm_id, and year quarter for Presenter Speech
            
            # Create document vector for each 
            tad_ps_q, tad_ps_a, tad_q_a = self.get_v_a_con(group)
            # save the TAD score for each firm, year, quarter
            result.append({'gvkey': names[0], 'year': names[1], 'quarter': names[2], 'tad_ps_q': tad_ps_q, 'tad_ps_a': tad_ps_a, 'tad_q_a': tad_q_a})
            # write the result to a csv file in append mode
            with open(os.path.join(os.getcwd(), "data", "TAD_score.csv"), "a") as f:
                f.write(f"{names[0]},{names[1]},{names[2]},{tad_ps_q},{tad_ps_a},{tad_q_a}\n")
        return result

    def add_filter_analyst_feature(self, row_doc, analyst_feature):
        # adjust the weight of the row_doc based on the analyst feature 
        # input: the row_doc and the analyst feature
        # output: the adjusted row_doc
        # get the analyst feature for each firm, year, quarter
        # fill NaN values with 0 in the analyst feature column
        row_doc[analyst_feature] = row_doc[analyst_feature].fillna(0)
        weights = row_doc[analyst_feature] + 1
        weights = weights / weights.sum()
        # adjust the weight of the row_doc based on the analyst feature
        row_doc[self.topics_] = row_doc.apply(lambda x: x[self.topics_] * weights/x['document_length'], axis=0)
        return row_doc
    
    # def compute_TAD(self, id2firms):
    #     # standardize the topic attention in id2firms
    #     # input: the id2firms data
    #     # output: the TAD for each firm
    #     wcd, wcl = self.read_word_contribution() # read the word contribution dictionary and the word contribution length
    #     # compute the TAD for each firm
    #     for topic in tqdm(wcd.keys(), desc="Computing TAD", bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}', colour="green"):
    #         # standardize the topic attention in id2firms
    #         id2firms[topic] = id2firms.apply(lambda x: x[topic] / (wcl[topic] * x['document_length']), axis=1)
    #     # aggregate the TAD from sentence level to section level
    #         # compute the TAD for each firm
    #     return id2firms 

In [ ]:
# load the narratives data by sentence_id, /outputs/scores/TF/combined_scores_TF.csv
narratives_path = os.path.join(os.getcwd(), "w2v_culture", "outputs", "scores", "TF", "combined_scores_TF.csv")
narratives = pd.read_csv(narratives_path)
narratives.head()